# Nutrition5k: resultados generales

Comparaciones entre familias. Primero la elección de la cabeza (etapa 1, backbone congelado, los 12 backbones);
después, un finalista por familia. Los detalles de cada familia están en `notebooks/<familia>/results.ipynb`.

In [ ]:
import os
from pathlib import Path

if Path("/workspace").is_dir():  # RunPod
    os.chdir("/root/thesis-multimodal-health-ai")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from nutrition5k.results import depth_steps, head_sizes, head_steps, load_heads, load_runs, stage2_runs

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

heads = load_heads(split="full")
print(f"{len(heads)} heads of {heads['backbone'].nunique()} backbones")

## Etapa 1: elección de la cabeza

Cada columna es el cambio del PMAE medio de validación al pasar de una cabeza a otra, medio sobre los backbones de
cada familia (`all backbones`: los 12). **Negativo = la segunda cabeza es mejor.** `seed SD` es la SD media entre las
semillas 0, 1 y 2 de la cabeza por defecto (`512 · single 256`): el ruido con el que comparar. Semilla 0, flip.

Nombre de cada cabeza: `trunk · cabeza`; `linear` = una capa lineal a los 5 objetivos, `single 256` = una capa
oculta de 256 compartida, `grouped 256` = una capa oculta de 256 por grupo de objetivos.

### 1. ¿Hace falta la capa oculta de 256?

In [ ]:
head_steps(heads, {
    "0: linear → single 256": ("0 · linear", "0 · single 256"),
    "512: linear → single 256": ("512 · linear", "512 · single 256"),
    "1024x1024: linear → single 256": ("1024x1024 · linear", "1024x1024 · single 256"),
})

### 2. Tamaño del trunk (512 → 1024x1024) y agrupación (single → grouped)

In [ ]:
head_steps(heads, {
    "512 → 1024x1024 (single 256)": ("512 · single 256", "1024x1024 · single 256"),
    "512 → 1024x1024 (grouped 256)": ("512 · grouped 256", "1024x1024 · grouped 256"),
    "single → grouped (512)": ("512 · single 256", "512 · grouped 256"),
    "single → grouped (1024x1024)": ("1024x1024 · single 256", "1024x1024 · grouped 256"),
    "512 · single → 1024x1024 · grouped": ("512 · single 256", "1024x1024 · grouped 256"),
})

### 3. Tamaño de la cabeza frente a la mejora

Media por familia: parámetros de la cabeza (miles), `head_pct` = % de los parámetros del modelo entero (backbone +
cabeza) que son de la cabeza, PMAE de validación, y `params ×` y `Δ val PMAE` respecto a `512 · single 256`.

In [ ]:
head_sizes(heads, ["512 · single 256", "512 · grouped 256", "1024x1024 · single 256", "1024x1024 · grouped 256"])

### Análisis (runs del stage 1 en `full`, 2026-10-31)

- **Capa oculta de 256: sí.** Sin trunk, pasar de lineal a `single 256` baja 5.2 puntos de media (3.6 a 7.7 por
  familia); con trunk 512, 1.0 (0.7 a 1.6). Con 1024x1024 ya casi no aporta (0.2), porque ese trunk ya da la
  no linealidad. Lo que hace falta es al menos una capa no lineal entre las features y la salida.
- **512 → 1024x1024:** −0.20 de media con `single 256` y −0.45 con `grouped 256` (de +0.07 a −0.57 por familia).
  Está por debajo del umbral de empate de 1 punto, pero en algunas familias supera la SD entre semillas (0.19 a
  0.46): puede ser una mejora real, pero pequeña.
- **Cabeza grande entera** (`512 · single 256` → `1024x1024 · grouped 256`): −0.30 de media (de +0.13 a −0.58 por
  familia).
- **single → grouped:** +0.15 con trunk 512 y −0.10 con 1024x1024, con signos distintos según la familia: sin efecto.
- **Tamaño:** `1024x1024 · single 256` tiene de 3.3 a 5.5 veces los parámetros de `512 · single 256`, y
  `1024x1024 · grouped 256` de 4 a 7.3 veces. En MobileNet la cabeza grande es casi la mitad del modelo (46 %, frente
  al 19 %) a cambio de ~0.5 puntos.
- Todo esto es con el backbone congelado, donde la cabeza es lo único que se adapta. Con fine-tuning la diferencia
  podría reducirse (hipótesis, sin medir); se comprueba en los finalistas.

## Etapa 2: profundidad del fine-tuning por familia

Media de los backbones de cada familia por nivel: `frozen` (0), `low` (1 stage; 6 bloques en los ViT), `high`
(2 stages; 10 bloques en los ViT) y `all`. `trainable_pct` es el % del backbone que se entrena; `vs_all` es el
cambio de PMAE de validación frente a entrenarlo todo, calculado backbone a backbone (negativo = mejor que `all`).

In [ ]:
stage = stage2_runs(load_runs(split="full"), heads)
depth_steps(stage)

## Finalistas

Un finalista por familia, con todas sus semillas (media; `val_sd` es la SD entre semillas). En `FINALISTS`, escribir
el nombre del run elegido de cada familia (con o sin `-s<semilla>`). Con `None` se toma la configuración con más
semillas (las finales de la etapa 3) y, entre ellas, la de menor validación media.
Los detalles de cada familia están en `notebooks/<familia>/results.ipynb`.

In [ ]:
import os
from pathlib import Path

if Path("/workspace").is_dir():  # RunPod
    os.chdir("/root/thesis-multimodal-health-ai")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

from nutrition5k.results import baseline_row, bar_targets, cost, finalists, load_runs, target_table

SPLIT = "pilot"         # "full" o "pilot"
FINALISTS = {"mobilenet": None, "hybrid_mobile": None, "plain_vit": None, "hierarchical": None}

final = finalists(load_runs(FINALISTS, SPLIT), FINALISTS)
final[["family", "backbone", "run", "seeds", "val_mean_pmae", "val_sd", "test_mean_pmae", "test_clean_mean_pmae",
       "params_M"]].round(2)

## Error por objetivo (validación) y por familia

In [ ]:
display(target_table(final, "backbone"))
bar_targets(final, "backbone", "Finalists: validation PMAE per target")

## Error frente a coste (Q7)

MACs y latencia en la CPU de esta máquina; la latencia que cuenta es la del iPhone.

In [ ]:
cost(final, "backbone", "Finalists: error vs compute (lower left is better)").round(2)